# IPSAE Interface Scoring

This notebook demonstrates `run_ipsae_scoring`, which scores a cofolded protein complex using the IPSAE method (Dunbrack 2025). It computes ipSAE, pDockQ2, LIS, pDockQ, and ipTM metrics from per-residue pLDDT and the PAE matrix.

The tool requires a `Structure` with pLDDT in the B-factor column and a PAE matrix at `structure.metrics['pae_matrix']`.

In [ ]:
import json
from pathlib import Path

from proto_tools.tools.structure_scoring.ipsae import (
    IPSAEScoringConfig,
    IPSAEScoringInput,
    run_ipsae_scoring,
)
from proto_tools.entities.structures import Structure
from proto_tools.entities.structures.structure import BFactorType

In [ ]:
# Load the example fixture (shared with pDockQ2)
fixture_dir = Path("../../pdockq2")
pdb_path = fixture_dir / "example_input_fixture.pdb"
pae_path = fixture_dir / "example_input_fixture_pae.json"

pae_matrix = json.loads(pae_path.read_text())
structure = Structure.from_file(
    pdb_path, b_factor_type=BFactorType.PLDDT, metrics={"pae_matrix": pae_matrix}
)

inputs = IPSAEScoringInput(
    structure=structure, binder_chain="A", target_chains=["B"]
)
print(f"Chains: {structure.get_chain_ids()}")
print(f"Binder: A | Target: B")

In [ ]:
# Run IPSAE scoring with default config
config = IPSAEScoringConfig()
result = run_ipsae_scoring(inputs, config)

print(f"ipsae:      {result.metrics.ipsae:.4f}")
print(f"pdockq2:    {result.metrics.pdockq2:.4f}")
print(f"lis:        {result.metrics.lis:.4f}")
print(f"pdockq:     {result.metrics.pdockq:.4f}")
print(f"iptm_d0chn: {result.metrics.iptm_d0chn:.4f}")

In [ ]:
# Inspect per-chain-pair breakdown
for row in result.metrics.chain_pair_results:
    print(
        f"{row.chain1}-{row.chain2} ({row.pair_type:4s}) | "
        f"ipSAE={row.ipsae:.4f}  pDockQ2={row.pdockq2:.4f}  "
        f"LIS={row.lis:.4f}  pDockQ={row.pdockq:.4f}  ipTM={row.iptm_d0chn:.4f}"
    )